In [1]:
import os
import json
import pandas as pd
import torch
from transformers import RobertaTokenizer, RobertaModel

In [ ]:
checkpoint_file = 'C:/Users/007pe/Desktop/checkpoint.json'
df_copy = pd.read_json(checkpoint_file, lines=True)

In [4]:
with open('C:/Users/007pe/Desktop/checkpoint.json', 'r') as f:
    checkpoint_data = json.load(f)

# Extract the 'data' field which contains the list of records
data_records = checkpoint_data['data']

# Create a DataFrame with the desired columns
df = pd.DataFrame(data_records)[['Speaker_party_name', 'text', 'tokens', 'embedding']]

# Display the DataFrame
print(df)

       Speaker_party_name                                               text  \
0        Liberal Democrat  1. What progress her Department has made on im...   
1            Conservative  The Government are on track to deliver their c...   
2        Liberal Democrat  It is clear that exit checks, which were scrap...   
3            Conservative  As I indicated in my original answer, we are o...   
4                  Labour  19. Given the situation at our border in Calai...   
...                   ...                                                ...   
670907       Conservative  I will have to check that point for the noble ...   
670908             Labour  My Lords, the Minister has referred a number o...   
670909       Conservative  The noble Lord is right that different aspects...   
670910             Labour  I thank noble Lords for a very interesting deb...   
670911       Conservative  My Lords, I wish the six or so stoic noble Pee...   

                                       

In [ ]:
# Checkpoint file
checkpoint_file = 'C:/Users/007pe/Downloads/checkpoint.json'
checkpoint_temp_file = 'C:/Users/007pe/Downloads/checkpoint_temp.json'  # Temporary file
original_file = 'C:/Users/007pe/Downloads/dataset_preprocessed.json'
original_df = pd.read_json(original_file, lines=True)

# Load checkpoint if it exists
if os.path.exists(checkpoint_file):
    print("Loading checkpoint...")
    with open(checkpoint_file, 'r') as f:
        checkpoint_data = json.load(f)

    # Retrieve original file path and data
    original_file = checkpoint_data['original_file']
    df_copy = pd.DataFrame(checkpoint_data['data'])
    start_index = len(df_copy)  # Start from the next row
    print(f"Checkpoint loaded with {start_index} rows. Original file: {original_file}")
else:
    print("No checkpoint found. Starting from the beginning.")
    start_index = 0
    df_copy = pd.DataFrame()  # Initialize an empty DataFrame
    print(f"Original dataset loaded with {len(df_copy)} rows.")

# Ensure start_index is within bounds
# if start_index >= len(original_df):
#     print("Start index out of bounds. Resetting to 0.")
#     start_index = 0

# Load pre-trained RoBERTa-large model and tokenizer
model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
roberta_model = RobertaModel.from_pretrained(model_name)

# Set the model to evaluation mode
roberta_model.eval()

def get_roberta_embeddings(tokens, model, tokenizer):
    """
    Generate RoBERTa embeddings for a list of tokens.
    - Joins tokens into a single string
    - Tokenizes text using RoBERTa tokenizer
    - Extracts embeddings from the last hidden layer
    - Returns average embedding as a list
    """
    # Join tokens into a single string
    text = " ".join(tokens)

    # Tokenize text
    encoded_input = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

    # Get embeddings
    with torch.no_grad():
        output = model(**encoded_input)

    # Extract last hidden state and average over tokens
    last_hidden_state = output.last_hidden_state  # Shape: [batch_size, seq_len, hidden_size]
    embedding = torch.mean(last_hidden_state, dim=1).squeeze().numpy()  # Average over tokens
    return embedding.tolist()  # Convert numpy array to list for JSON serialization

# Load the original file for remaining rows

print(f"Length of original dataset: {len(original_df)}")

# Process data in chunks
chunk_size = 1000
processed_chunks = [df_copy]  # Start with the checkpointed data

for i in range(start_index, len(original_df), chunk_size):
    print(f"Processing chunk {i} to {i + chunk_size}...")

    # Process the current chunk
    chunk = original_df.iloc[i:i + chunk_size].copy()
    chunk['embedding'] = chunk['tokens'].apply(lambda x: get_roberta_embeddings(x, roberta_model, tokenizer))

    # Debugging: Print the first row of the chunk to verify embeddings
    print(f"First row of chunk {i}:")
    print(chunk.iloc[0])

    processed_chunks.append(chunk)

    # Save checkpoint to a temporary file first
    checkpoint_df = pd.concat(processed_chunks, ignore_index=True)
    checkpoint_data = {
        'original_file': original_file,
        'data': checkpoint_df.to_dict(orient='records')
        }
    with open(checkpoint_temp_file, 'w') as f:
        json.dump(checkpoint_data, f)

    # Rename the temporary file to the checkpoint file
    os.replace(checkpoint_temp_file, checkpoint_file)
    print(f"Checkpoint saved at row {i + chunk_size}.")

# Combine all processed chunks into a single DataFrame
df_copy = pd.concat(processed_chunks, ignore_index=True)

# Save the final DataFrame
df_copy.to_json('C:/Users/007pe/Downloads/ready_data.json', orient='records', lines=True)
print("Embedding generation complete. Final DataFrame saved.")

In [5]:
df_copy2 = df

In [8]:
from collections import defaultdict

party_ideology_mapping = {
    'Conservative': 'Centre-right to right-wing',
    'Labour': 'Centre-left',
    'Liberal Democrat': 'Centre to centre-left',
    'Scottish National Party': 'Centre-left to left-wing',
    'Crossbench': 'Non-partisan',
    'Democratic Unionist Party': 'Right-wing',
    'Green Party': 'Left-wing',
    'Plaid Cymru': 'Centre-left to left-wing',
    'Independent': 'Independent',
    'Bishops': 'Cleric',
    'Social Democratic & Labour Party': 'Centre-left',
    'Non-affiliated': 'Non-partisan',
    'Independent;Conservative': 'Centre-right to right-wing',
    'Labour;Non-affiliated': 'Centre-left',
    'Ulster Unionist Party': 'Right-wing',
    'UK Independence Party': 'Right-wing to far-right',
    'Alliance': 'Centre to centre-left',
    'Non-affiliated;Crossbench': 'Non-partisan',
    'Scottish National Party;Independent': 'Centre-left to left-wing',
    'Non-affiliated;Conservative': 'Centre-right to right-wing',
    'Non-affiliated;Labour': 'Centre-left',
    'Conservative;Independent': 'Centre-right to right-wing',
    'Plaid Cymru;Independent': 'Centre-left to left-wing',
    'Conservative;Crossbench': 'Centre-right to right-wing',
    'Scottish National Party;Alba Party': 'Centre-left to left-wing',
    'The Independent Group for Change': 'Centre-left',
    'Independent Labour': 'Independent',
    'Labour Independent': 'Centre-left',
    'Conservative;Non-affiliated': 'Centre-right to right-wing',
    'Labour;Independent': 'Centre-left',
    'Liberal Democrat;Non-affiliated': 'Centre to centre-left',
    'Change UK - The Independent Group;The Independent Group for Change': 'Centre-left',
    'Independent;Liberal Democrat': 'Centre to centre-left',
    'Liberal Democrat;Crossbench': 'Centre to centre-left',
    'Independent Ulster Unionist': 'Independent',
    'Independent Liberal Democrat': 'Independent',
    'Independent;Crossbench': 'Independent',
    'Conservative;Labour': 'Centre-right to right-wing',  # Rare combination, likely an error
    'Independent Social Democrat': 'Independent',
    'Respect': 'Left-wing to far-left',
    'The Independent Group for Change;Change UK - The Independent Group': 'Centre-left',
    'UK Independence Party;Non-affiliated': 'Right-wing to far-right',
    'Independent Labour;Non-affiliated': 'Centre-left',
    'Conservative;Independent;Liberal Democrat': 'Centre-right to right-wing',
    'Conservative;Liberal Democrat': 'Centre-right to right-wing',  # Rare combination, likely an error
    'Conservative;Non-affiliated;Crossbench': 'Centre-right to right-wing',
    'Conservative;Independent Conservative;Non-affiliated': 'Centre-right to right-wing',
    'UK Independence Party;Independent': 'Right-wing to far-right',
    'Liberal Democrat;Non-affiliated;Crossbench': 'Centre to centre-left',
    'Crossbench;Labour': 'Non-partisan'
}

mapped_dict = defaultdict(set)

for party, ideology in party_ideology_mapping.items():
    mapped_dict[ideology].update([party])

def map_values(category):
    for ideology, parties in mapped_dict.items():
        if category in parties:
            return ideology
    return None

df_copy2['Speaker_party_name'] = df_copy2['Speaker_party_name'].apply(map_values)
print(df_copy2)
print(df_copy2['Speaker_party_name'].value_counts())

                Speaker_party_name  \
1       Centre-right to right-wing   
2            Centre to centre-left   
3       Centre-right to right-wing   
4                      Centre-left   
5       Centre-right to right-wing   
...                            ...   
670906                 Centre-left   
670907  Centre-right to right-wing   
670908                 Centre-left   
670909  Centre-right to right-wing   
670910                 Centre-left   

                                                embedding  
1       [-0.017255421727895737, 0.04112221300601959, 0...  
2       [-0.019146282225847244, 0.01313154585659504, -...  
3       [-0.10002964735031128, -0.05257001519203186, 0...  
4       [0.08510342240333557, 0.014632830396294594, -0...  
5       [-0.003270169720053673, 0.11290641874074936, 0...  
...                                                   ...  
670906  [-0.16485783457756042, -0.13624031841754913, 0...  
670907  [-0.11614319682121277, -0.010834960266947746, ...  
670

In [9]:
df_copy2.to_json('C:/Users/007pe/Downloads/ready_data.json', orient='records', lines=True)